# 05. Validarea urmelor sintetice cu 100 de experimente

Notebook-ul testează un model de trace mai apropiat de o măsurătoare reală:

\[
T_i[t] = A_i L_i[t-\tau_i] + B_i[t] + U_i[t] + \epsilon_i[t].
\]

`L` este semnalul de leakage generat din modelul ECDSA existent; `A` modifică amplitudinea, `\tau` introduce un deplasament temporal, `B(t)` modelează variația lentă a baseline-ului, `U(t)` modelează activitatea necorelată cu nonce-ul, iar `\epsilon(t)` este zgomot gaussian independent.

Experimentul rulează exact 100 de instanțe independente, cu seed-urile `1000...1099`. Fiecare instanță folosește un eșantion ECDSA diferit din datasetul proiectului și păstrează separat componentele pentru verificarea numerică.

**Scopul acestui notebook:** validarea generatorului de traces și măsurarea cât de mult rămâne vizibil semnalul de leakage. Rezultatele nu sunt tratate ca măsurători hardware ESP32.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "informatica").is_dir() and (p / "notebooks").is_dir()
)

ECDSA_DIR = PROJECT_ROOT / "informatica" / "ECDSA"
SIM_DIR = PROJECT_ROOT / "informatica" / "simulator"
ARTIFACTS_DIR = PROJECT_ROOT / "informatica" / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
RAW_DIR = RESULTS_DIR / "raw"

for path in (ECDSA_DIR, SIM_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

for directory in (ARTIFACTS_DIR, FIGURES_DIR, TABLES_DIR, RAW_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import random
from dataclasses import asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataset_generator3 import CURVE, load_ground_truth, load_public_dataset
from ecdsa_leakage_model4 import generate_clean_leakage
from realistic_trace import RealisticTraceConfig, generate_trace, evaluate_trace


## 1. Configurația experimentală

Valorile de mai jos sunt parametri de simulare, nu valori pretinse ca fiind măsurate pe un ESP32. Ele pot fi schimbate într-un sweep separat. În experimentul principal păstrăm `trace_length = 256`, `leaked_bits = 12` și `noise_sigma = 0.15`, astfel încât partea de leakage să rămână compatibilă cu proiectul existent.

In [ ]:
config = RealisticTraceConfig(
    trace_length=256,
    noise_sigma=0.15,
    amplitude_mean=1.0,
    amplitude_std=0.08,
    min_amplitude=0.70,
    max_amplitude=1.30,
    max_shift=4,
    baseline_offset_std=0.50,
    baseline_slope_std=0.60,
    baseline_sine_std=0.35,
    unrelated_activity_count=(1, 3),
    unrelated_activity_amplitude=1.50,
    unrelated_activity_width=(2.0, 8.0),
    leakage_alpha=1.0,
    leakage_mode="HW",
    seed=1000,
)

config.validate()
display(pd.DataFrame([
    ["trace_length", config.trace_length, "eșantioane / urmă"],
    ["noise_sigma", config.noise_sigma, "abaterea standard a zgomotului"],
    ["amplitude", f"N({config.amplitude_mean}, {config.amplitude_std**2:.4f}) limitat la [{config.min_amplitude}, {config.max_amplitude}]", "gain al leakage-ului"],
    ["shift", f"[-{config.max_shift}, +{config.max_shift}]", "eșantioane"],
    ["baseline", f"offset={config.baseline_offset_std}, slope={config.baseline_slope_std}, sine={config.baseline_sine_std}", "variație lentă"],
    ["unrelated", f"{config.unrelated_activity_count[0]}...{config.unrelated_activity_count[1]} impulsuri", "activitate independentă de nonce"],
    ["seeds", "1000...1099", "100 experimente"],
], columns=["parametru", "valoare", "rol"]))

## 2. Datasetul și semnalul de leakage

Folosim aceleași `public_dataset.json` și `oracle_dataset.json` din proiect. Pentru fiecare `sample_id` calculăm semnalul curat cu funcția existentă `generate_clean_leakage()`. Cheia privată și nonce-ul complet rămân date oracle pentru validare; ele nu intră în trace-ul observabil.

In [ ]:
public_records = load_public_dataset(ARTIFACTS_DIR / "public_dataset.json")
oracle = load_ground_truth(ARTIFACTS_DIR / "oracle_dataset.json")

if len(public_records) < 100 or len(oracle["samples"]) < 100:
    raise ValueError("Proiectul trebuie să conțină cel puțin 100 de eșantioane pentru acest test.")

clean_by_id = {}
for record in public_records[:100]:
    sample_id = int(record["sample_id"])
    nonce = int(oracle["samples"][str(sample_id)]["nonce_k"])
    clean_by_id[sample_id] = np.asarray(
        generate_clean_leakage(
            nonce,
            config.trace_length,
            mode=config.leakage_mode,
            alpha=config.leakage_alpha,
        ),
        dtype=float,
    )

assert len(clean_by_id) == 100
print(f"Pregătite {len(clean_by_id)} semnale de leakage pentru test.")

## 3. Un experiment individual: descompunerea trace-ului

Pentru o urmă păstrăm separat cele cinci contribuții. Verificarea algebrică este directă:

\[
T - B - U - \epsilon = A\,L_{shifted}.
\]

Dacă această identitate nu este satisfăcută numeric, generatorul are o eroare de implementare.

In [ ]:
example_id = 0
example_seed = config.seed + example_id
example_rng = random.Random(example_seed)
example_components = generate_trace(clean_by_id[example_id], config, example_rng)
example_metrics = evaluate_trace(example_components)

reconstructed = (
    example_components.amplitude_scaled
    + example_components.baseline
    + example_components.unrelated_activity
    + example_components.noise
)

print("Eroare maximă de reconstrucție:", float(np.max(np.abs(example_components.trace - reconstructed))))
print("Amplitudine:", example_components.amplitude)
print("Shift:", example_components.shift)
print("SNR estimat (dB):", example_metrics["snr_db"])


In [ ]:
x = np.arange(config.trace_length)

plt.figure(figsize=(12, 5))
plt.plot(x, example_components.trace, label="trace observat")
plt.plot(x, example_components.amplitude_scaled, label="leakage × amplitudine")
plt.plot(x, example_components.baseline, label="baseline")
plt.plot(x, example_components.unrelated_activity, label="activitate necorelată")
plt.plot(x, example_components.noise, label="zgomot")
plt.xlabel("t")
plt.ylabel("amplitudine")
plt.title("Descompunerea unei urme sintetice")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "trace-validation-example.png", dpi=160)
plt.show()


## 4. Cele 100 de experimente

Fiecare experiment are propriul seed. Astfel, aceeași urmă poate fi regenerată exact, iar seed-ul este suficient pentru a identifica configurația aleatoare a amplitudinii, shift-ului, baseline-ului, activității necorelate și zgomotului.

Măsurăm:

- `snr_db`: raportul dintre puterea leakage-ului după amplitudine și puterea totală a perturbărilor;
- `leakage_correlation`: corelația dintre componenta de leakage și urma observată;
- RMS pentru baseline, activitate necorelată și zgomot;
- `reconstruction_relative_error`: verifică implementarea ecuației trace-ului.

Aceste valori testează generatorul. Nu sunt acuratețe CNN și nu sunt rată de recuperare HNP.

In [ ]:
rows = []
traces = []
leakages = []
baselines = []
unrelated = []
noise = []
amplitudes = []
shifts = []

for experiment_id, sample_id in enumerate(list(clean_by_id)[:100], start=1):
    seed = config.seed + experiment_id - 1
    rng = random.Random(seed)
    components = generate_trace(clean_by_id[sample_id], config, rng)
    metrics = evaluate_trace(components)

    rows.append({
        "experiment_id": experiment_id,
        "seed": seed,
        "sample_id": sample_id,
        "amplitude": components.amplitude,
        "shift": components.shift,
        "noise_sigma": config.noise_sigma,
        **metrics,
    })
    traces.append(components.trace)
    leakages.append(components.leakage)
    baselines.append(components.baseline)
    unrelated.append(components.unrelated_activity)
    noise.append(components.noise)
    amplitudes.append(components.amplitude)
    shifts.append(components.shift)

results_df = pd.DataFrame(rows)
results_df.to_csv(TABLES_DIR / "trace-validation-100.csv", index=False)
np.savez_compressed(
    RAW_DIR / "trace-validation-100.npz",
    traces=np.asarray(traces),
    leakages=np.asarray(leakages),
    baselines=np.asarray(baselines),
    unrelated_activity=np.asarray(unrelated),
    noise=np.asarray(noise),
    amplitudes=np.asarray(amplitudes),
    shifts=np.asarray(shifts),
)

metadata = {
    "experiment_count": 100,
    "seed_range": [1000, 1099],
    "trace_length": config.trace_length,
    "leaked_bits": 12,
    "formula": "T[t] = A * L[t - tau] + B[t] + U[t] + epsilon[t]",
    "config": asdict(config),
    "outputs": {
        "csv": str(TABLES_DIR / "trace-validation-100.csv"),
        "npz": str(RAW_DIR / "trace-validation-100.npz"),
        "example_figure": str(FIGURES_DIR / "trace-validation-example.png"),
    },
}
(ARTIFACTS_DIR / "trace_validation_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

print(f"Au fost rulate {len(results_df)} experimente.")
display(results_df.head(10))


## 5. Criterii de integritate

Pentru acest test, condițiile `PASS` sunt mecanice:

1. trebuie să existe exact 100 de rezultate;
2. seed-urile trebuie să fie exact `1000...1099`;
3. amplitudinea trebuie să rămână în limitele configurate;
4. shift-ul trebuie să rămână în `[-4, +4]`;
5. identitatea de reconstrucție trebuie să aibă eroare numerică practic zero;
6. fiecare experiment trebuie să conțină zgomot și, în configurația implicită, activitate necorelată;
7. rularea cu aceleași seed-uri trebuie să producă aceleași urme.

Nu impunem un prag universal pentru SNR sau corelație. Aceste praguri depind de nivelul de stres pe care îl doriți pentru CNN/HNP și trebuie stabilite separat înainte de un benchmark de performanță.

In [ ]:
assert len(results_df) == 100
assert results_df["seed"].tolist() == list(range(1000, 1100))
assert results_df["amplitude"].between(config.min_amplitude, config.max_amplitude).all()
assert results_df["shift"].between(-config.max_shift, config.max_shift).all()
assert results_df["reconstruction_relative_error"].max() < 1e-12
assert results_df["noise_rms"].gt(0).all()
assert results_df["unrelated_rms"].ge(0).all()

print("PASS: integritatea celor 100 de experimente este validată.")


## 6. Rezumat statistic al celor 100 de urme

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "amplitude", "shift", "baseline_rms", "unrelated_rms",
        "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"
    ],
    "mean": [results_df[c].mean() for c in ["amplitude", "shift", "baseline_rms", "unrelated_rms", "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"]],
    "median": [results_df[c].median() for c in ["amplitude", "shift", "baseline_rms", "unrelated_rms", "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"]],
    "min": [results_df[c].min() for c in ["amplitude", "shift", "baseline_rms", "unrelated_rms", "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"]],
    "max": [results_df[c].max() for c in ["amplitude", "shift", "baseline_rms", "unrelated_rms", "noise_rms", "nuisance_rms", "snr_db", "leakage_correlation"]],
})

display(summary)

In [ ]:
fig = plt.figure(figsize=(11, 4.5))
plt.hist(results_df["snr_db"], bins=12)
plt.xlabel("SNR (dB)")
plt.ylabel("număr de experimente")
plt.title("Distribuția SNR pentru 100 de urme")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "trace-validation-snr.png", dpi=160)
plt.show()

fig = plt.figure(figsize=(11, 4.5))
plt.scatter(results_df["shift"], results_df["leakage_correlation"], alpha=0.8)
plt.xlabel("shift (eșantioane)")
plt.ylabel("corelație cu leakage")
plt.title("Corelația leakage vs. deplasare temporală")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "trace-validation-shift-correlation.png", dpi=160)
plt.show()


## 7. Cum se leagă testul de restul proiectului

Acest notebook trebuie privit ca o etapă de validare a datelor de intrare. În proiectul existent, CNN-ul primește traces și produce prefixe de nonce, iar etapa HNP folosește acele prefixe împreună cu `r`, `s` și `z`. Prin urmare, după ce generatorul trece testul de 100 de experimente, un benchmark separat poate compara acuratețea CNN pentru traces simple și pentru traces cu perturbări.

Testul de mai sus nu modifică splitul `96 / 24 / 40` din protocolul CNN și nu schimbă criteriul de validare HNP. De asemenea, rezultatele rămân sintetice până la apariția măsurătorilor fizice ESP32.